# Train a Custom YOLO Model — Simple Dataset Preparation


This notebook is designed to help you train your own YOLO models from scratch or fine-tune existing ones.  
You can use either:

- **Local datasets** (e.g., in YOLO format stored on your Google Drive or GitHub)
- **Datasets from Roboflow**, which can be easily imported via a download link
- **Example dataset**, from a linked github repository

The workflow includes:
- Loading and organizing your dataset
- Writing a custom `.yaml` config file
- Launching training with the `ultralytics` YOLO implementation
- (Optional) Exporting and evaluating your trained model

This is ideal for training models on custom objects — whether you're working with animals, vehicles, tools, or underwater footage.

---

Make sure your dataset is in the correct YOLO structure:

```
dataset/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
├── test/   # optional
│   ├── images/
│   └── labels/
└── data.yaml
```

# Import libraries

In [ ]:
import glob
import os
import random
import shutil
import sys
import time
from pathlib import Path

from google.colab import drive, runtime
from IPython.display import Image, display


In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# Run this before importing NumPy, SciPy, Ultralytics, or dataset-fixer.
%pip install --upgrade ultralytics
%pip install "dataset-fixer @ git+https://github.com/mooch443/dataset-fixer.git"

import ultralytics
ultralytics.checks()

from ultralytics import YOLO
from dataset_fixer import Dataset as FixedDataset
import numpy as np


In [ ]:
# The dataset selector loads its helper directly and is safe to run independently.
print('Dataset helper will be loaded by the selector cell.')


### 🔗 Connect to Your Google Drive

Google Colab is a cloud-based Python environment that lets you run code in your browser, with free access to GPUs.  

To access your datasets or save model outputs, you’ll need to connect Colab to your Google Drive.
This allows you to read and write files directly from your Drive, making it easier to store large datasets or export trained models.

Run the cell below to authorize access.

In [6]:
# Mount google drive
drive.mount("/content/drive/")

Mounted at /content/drive/


# Load data

Use the interactive widget below to choose your dataset source. You can select one of:

### 1. Roboflow
- Paste your **Roboflow API key**, **workspace**, **project**, **version**, and **export format** (e.g., `yolov11`).
- Tip: Usually copied from **Roboflow → Export → Show download code** on your project page.

### 2. Google Drive link (shared `.zip`)
- Copy the **Drive share URL** of your `.zip` (set sharing to **Anyone with the link**).
- Paste it into the widget and confirm to download and extract.

### 3. Example dataset from GitHub (Hexbugs)
- Select **Hexbugs** to load a small, YOLO-formatted example dataset for quick testing.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

helper_candidates = [
    Path.cwd() / 'utils' / 'datasets.py',
    Path('/content/train-custom-YOLO-Colab/utils/datasets.py'),
]
helper_path = next((path.resolve() for path in helper_candidates if path.is_file()), None)

if helper_path is None:
    repository_root = Path('/content/train-custom-YOLO-Colab')
    subprocess.run([
        'git', 'clone',
        'https://github.com/albiangela/train-custom-YOLO-Colab.git',
        str(repository_root),
    ], check=True)
    helper_path = repository_root / 'utils' / 'datasets.py'

spec = importlib.util.spec_from_file_location('_yolo_dataset_selector', helper_path)
datasets_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = datasets_module
spec.loader.exec_module(datasets_module)

workspace_dataset_root = Path('/content/datasets')
print('Dataset selector loaded from:', helper_path)
datasets_module.launch_dataset_selector(globals(), dataset_root=workspace_dataset_root)


# Validate, fix, and split the dataset

This version keeps every original class and uses `dataset-fixer` for validation,
the reproducible train/validation/test split, canonical export, reports, and
`data.yaml` generation. The source dataset is never modified.


In [ ]:
import yaml
from types import SimpleNamespace

dataset = SimpleNamespace(location='/content/datasets/', name=name, version=1)
dataset_root = os.path.join(dataset.location, name)
source_yaml = os.path.join(dataset_root, 'data.yaml')
out_dir = dataset_root + '-dataset-fixed'

assert os.path.isdir(os.path.join(dataset_root, 'train')), dataset_root
assert os.path.isfile(source_yaml), source_yaml

# Build a portable compatibility YAML without modifying the downloaded source.
# This also repairs Roboflow exports that contain paths such as ../train/images.
with open(source_yaml, 'r') as stream:
    fixer_yaml_data = yaml.safe_load(stream) or {}
fixer_yaml_data['path'] = os.path.abspath(dataset_root)
for split_key in ('train', 'val', 'valid', 'validation', 'test'):
    fixer_yaml_data.pop(split_key, None)
for yaml_key, folder_candidates in {
    'train': ('train',),
    'val': ('val', 'valid', 'validation'),
    'test': ('test',),
}.items():
    for folder_name in folder_candidates:
        if os.path.isdir(os.path.join(dataset_root, folder_name, 'images')):
            fixer_yaml_data[yaml_key] = f'{folder_name}/images'
            break

fixer_source_yaml = dataset_root + '-dataset-fixer-input.yaml'
with open(fixer_source_yaml, 'w') as stream:
    yaml.safe_dump(fixer_yaml_data, stream, sort_keys=False, allow_unicode=True)

print('Dataset:', dataset_root)
print('dataset-fixer input:', fixer_source_yaml)
print('Fixed output:', out_dir)


In [ ]:
fixed_dataset = FixedDataset.open(
    fixer_source_yaml,
    errors='skip',
    progress=True,
)

split_plan = fixed_dataset.split(
    {'train': 0.7, 'val': 0.2, 'test': 0.1},
    seed=42,
    visualize=True,
    progress=True,
)

overwrite_output = False
if os.path.exists(out_dir):
    if overwrite_output:
        shutil.rmtree(out_dir)
    else:
        raise FileExistsError(
            f'{out_dir} already exists. Delete it or set overwrite_output=True before rerunning.'
        )

exported = split_plan.export(
    destination=out_dir,
    visualize=True,
    progress=True,
)
exported.assert_trainable()

zip_path = shutil.make_archive(
    out_dir,
    'zip',
    root_dir=os.path.dirname(out_dir),
    base_dir=os.path.basename(out_dir),
)
print('data.yaml:', exported.data_yaml)
print('ZIP written to:', zip_path)


### Define output path

In [ ]:
### Change path to your folder
REMOTE_URL = "/content/drive/MyDrive/models/" + name
HOME = "/content/datasets/"

# Change to HOME directory
%cd {HOME}

# Import os and create the folder if it doesn't exist
import os

if not os.path.exists(REMOTE_URL):
    os.makedirs(REMOTE_URL)
    print(f"Directory '{REMOTE_URL}' created.")
else:
    print(f"Directory '{REMOTE_URL}' already exists.")

# Training: Parameters

In [ ]:
# Change to home directory
%cd {HOME}

# ---- User-defined Settings ----
resolution = 1080                # Image resolution for training
epochs = 300                     # Number of training epochs
batch_size = 6                   # Batch size
base_model = "yolo26s"         # Choose model variant. Options: "yolo11n-pose", "yolo11n-seg", "yolo11n" etc.


# ---- Auto-detect task type ----
if "-seg" in base_model:
    task = "segment"
elif "-pose" in base_model:
    task = "pose"
else:
    task = "detect"

# ---- 🔧 Training Settings ----
common_settings = {
    "translate": 0.05,       # Maximum image translation as data augmentation (in % of image size)
    "mixup": 0.1,          # MixUp blending factor for image mixing (usually low for object detection)
    "copy_paste": 0.3,      # Probability of using Copy-Paste augmentation (object pasting)
    "scale": 0.3,           # Random scaling of images for augmentation
    "mosaic": 0.5,             # Enable Mosaic augmentation (combines 4 images into 1)
    "close_mosaic": 10,      # Number of epochs before disabling mosaic for better fine-tuning
    "line_width": 1,         # Line width for label visualization
    "nms": True,             # Apply Non-Maximum Suppression during inference
    "plots": True,           # Save training plots (loss, mAP, etc.)
    "cache": "disk",         # Caching mode: "disk" to speed up I/O
    "single_cls": False,     # If True, treat all objects as one class (for class-agnostic detection)
    "amp": True,             # Enable automatic mixed precision (reduces memory, speeds up training)
    "augment": True,        # If True, applies augmentation at inference time
    "workers": 16,            # Number of dataloader workers (adjust depending on your CPU)
    "multi_scale": False,
    "hsv_h": 0.015,
    "hsv_s": 0.5,
    "hsv_v": 0.4,
    "iou": 0.1,
    "bgr":0.1,
    "warmup_epochs": 2,
    "save_crop": True,
    "end2end": False,  # Turn of end-to-end
    "agnostic_nms": True


}


# Modify task-specific augmentations
if task == "detect":
    common_settings.update({
        "degrees": 10,       # Allow full rotation
        "flipud": 0.0,       # Vertical flip probability
        "fliplr": 0.0        # Horizontal flip probability
    })
else:
    common_settings.update({
        "degrees": 0,         # No rotation for pose/seg
        "flipud": 0.0,
        "fliplr": 0.0
    })

# Print CLI training parameters
parms = " ".join([f"{k}={v}" for k, v in common_settings.items()])
print("🔧 Training params:", parms)

# ---- 🗂️ Model Output Naming ----
from datetime import datetime
now = datetime.now()
date_string = now.strftime("%Y-%m-%d-%H") + "_" + dataset.name.replace(" ", "-") + "-" + str(dataset.version)

project = f"{resolution}-{base_model}"
if common_settings["mosaic"] > 0:
    project += "-mosaic"

# if sharkcam:
#     project += "-sharkcam"  # Add logic if needed

# ---- 🧠 Model Weights Source ----
model = base_model  # or path to a pretrained model
print(f"🧪 resolution={resolution} | project={project} | date_string={date_string}")
print(f"📦 model={model} | base_model={base_model} | task={task}")

# ---- 🔒 Safety Check ----
import os
assert model == base_model or os.path.exists(model + ".pt"), f"Model path not found: {model}.pt"

# Training: Run Command

In [ ]:
    # Change to your working directory
%cd {HOME}

# ---- Launch YOLO training ----
yolo_cmd = f"""
yolo task={task} \
     mode=train \
     resume=False \
     model={model}.pt \
     data={out_dir}/data.yaml \
     device=0 \
     name={date_string} \
     project={project} \
     epochs={epochs} \
     imgsz={resolution} \
     batch={batch_size} \
     patience=0 \
     visualize=True \
     {parms}
"""

# ▶️ Run the command
!{yolo_cmd}

#5) Locate last trained model

In [ ]:
# Change to the working directory
%cd {HOME}

runs_root = f"{dataset.location}/runs/{task}/{project}"

# List all subdirectories in the project folder
all_subdirs = [os.path.join(runs_root, d) for d in os.listdir(runs_root)]
all_subdirs = [d for d in all_subdirs if os.path.isdir(d)]

# Keep only those that contain a trained model
all_subdirs = [d for d in all_subdirs if os.path.exists(os.path.join(d, "weights", "last.pt"))]


# Get the most recently modified subdirectory
latest_subdir = max(all_subdirs, key=os.path.getmtime)

# Construct the full path to the latest run
# full_path = HOME + "/" + latest_subdir

print(project)
print(latest_subdir)
# print(full_path)

# Save training parameters to a parms.txt
!echo "{parms}" > {latest_subdir}/parms.txt

### Select and Save Best YOLO Model Based on mAP Metrics

In [ ]:

import pandas as pd

# Check if the best model weights file exists
print(os.path.exists(os.path.join(latest_subdir, "weights", "best.pt")))

# Load training results CSV
csv = pd.read_csv(os.path.join(latest_subdir, "results.csv"))

# Strip whitespace from column names
csv.columns = [c.strip() for c in csv.columns]

# Compute best epoch score (seg or det)
if "metrics/mAP50-95(M)" in csv.columns:
    combined = (csv["metrics/mAP50-95(M)"] * 0.9 + csv["metrics/mAP50(M)"] * 0.1) + \
               (csv["metrics/mAP50-95(B)"] * 0.9 + csv["metrics/mAP50(B)"] * 0.1)
    index = combined.argmax()
    best_map50_95 = float(csv["metrics/mAP50-95(M)"].values[index])
    best_map50 = float(csv["metrics/mAP50(M)"].values[index])
else:
    combined = (csv["metrics/mAP50-95(B)"] * 0.9 + csv["metrics/mAP50(B)"] * 0.1)
    index = combined.argmax()
    best_map50_95 = float(csv["metrics/mAP50-95(B)"].values[index])
    best_map50 = float(csv["metrics/mAP50(B)"].values[index])

# Copy best.pt to /content with informative name
from_path = os.path.join(latest_subdir, "weights", "best.pt")
to_name = f"{project}-{date_string}-mAP5095_{best_map50_95:.5f}-mAP50_{best_map50:.5f}.pt"
to_path = os.path.join("/content", to_name)

print("copying from", from_path, "to", to_path)
!cp "{from_path}" "{to_path}"
!rsync --progress "{to_path}" "{REMOTE_URL}/"

# Zip the whole run folder (latest_subdir) and upload
run_name = os.path.basename(latest_subdir)  # e.g. 2026-01-31-11_panabat-1
zip_path = os.path.join("/content", f"{project}-{run_name}.zip")

print("zipping", latest_subdir, "->", zip_path)
!zip -r "{zip_path}" "{latest_subdir}"
!rsync --progress "{zip_path}" "{REMOTE_URL}/"

In [ ]:
!rsync --progress {to_path} {REMOTE_URL}/


### Training results plot

In [ ]:
# Change working directory to HOME
%cd {HOME}

# Display the training results plot (e.g. loss and metrics curves)
Image(filename=f'{latest_subdir}/results.png', width=1200)

### Sample batch of validation predictions

In [ ]:
# Change working directory to HOME
%cd {HOME}

# Display a sample batch of validation predictions (visual output of model)
Image(filename=f'{latest_subdir}/val_batch0_pred.jpg', width=600)

# 6) Validate Custom Model

This step runs **model validation** using the best trained checkpoint (`best.pt`) on the validation dataset defined in `data.yaml`. It evaluates the model's performance using standard YOLO metrics, such as:

- **mAP50**: mean Average Precision at IoU threshold 0.5
- **mAP50-95**: mean AP across IoU thresholds from 0.5 to 0.95
- **Precision & Recall** for each class

The validation results will be saved inside the specified project folder and include:

- A `results.png` file with training/validation curves
- A `confusion_matrix.png` for classification performance
- A `val_batch0_pred.jpg` showing predicted bounding boxes on a sample batch

You can use these visual and quantitative outputs to assess if the model generalizes well to unseen data. [link text](https://)

In [ ]:
# Change working directory to HOME
%cd {HOME}

# Display the training results plot (e.g. loss and metrics curves)
Image(filename=f'{latest_subdir}/results.png', width=1200)

# 7) Run Inference on Validation Images

This step performs **inference (prediction)** using the best trained YOLO model (`best.pt`) on the validation image set. It is useful to **visually inspect how the model performs** on real images after training.

What this does:

- Removes any existing `predict` folder to avoid clutter or overwriting previous predictions
- Runs YOLO in `predict` mode using:
  - The best model checkpoint
  - Images from the validation set
  - A low confidence threshold (`conf=0.1`) to allow more predictions for visual inspection
  - The specified image size (`imgsz`)
- Saves predicted images (with boxes, masks, or keypoints depending on the task) in a new folder under the project directory: `runs/predict`

This is especially helpful for qualitatively checking the model's detection performance, spotting failure cases, or selecting images for visualization or presentations.

In [ ]:
%cd {HOME}

best_ckpt = os.path.join(latest_subdir, "weights", "best.pt")
data_yaml = os.path.join(out_dir, "data.yaml")   # <-- fix: use out_dir

assert os.path.exists(best_ckpt), best_ckpt
assert os.path.exists(data_yaml), data_yaml

!yolo task={task} mode=val model="{best_ckpt}" data="{data_yaml}" project="{project}" imgsz={resolution} line_width=1

### Zip and Save Prediction Results

This step creates a ZIP archive of the prediction results generated in the previous step. The archive is saved in your home directory and named using the training subdirectory name (to make it easy to track which model it came from).

This makes it simple to download, share, or upload the predictions for external use (e.g., for presentations, manual inspection, or further analysis).

In [ ]:
# Change working directory to HOME
%cd {HOME}

best_ckpt   = os.path.join(latest_subdir, "weights", "best.pt")
valid_imgs  = os.path.join(out_dir, "val", "images")   # <-- use out_dir (your dataset root)
pred_dir    = os.path.join(project, "predict")           # project/name

print("Model:", best_ckpt)
print("Source:", valid_imgs)
print("Will write to:", pred_dir)

# Remove any previous YOLO prediction results (safe delete)
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)

# Run YOLO prediction on validation images using the best model checkpoint
!yolo task={task} mode=predict model="{best_ckpt}" project="{project}" name="predict" conf=0.1 source="{valid_imgs}" save=True imgsz={resolution} line_width=1

# 10) Display Sample Predictions

This step randomly selects and displays 5 predicted images from the `predict` folder.

Each image includes the model's output (e.g., bounding boxes, masks, or keypoints) overlaid on the validation images.  
It provides a quick **visual inspection** of model performance across different examples.  

This qualitative check helps identify:
- How well the model localizes objects
- Possible false positives or negatives
- Class confusion or missed detections

In [ ]:
# Find newest predict* folder for this task/project
pred_root = f"/content/datasets/runs/{task}/{project}"
pred_candidates = [p for p in glob.glob(f"{pred_root}/predict*") if os.path.isdir(p)]
assert pred_candidates, f"No predict folder found in: {pred_root}"
pred_dir = max(pred_candidates, key=os.path.getmtime)

# Randomly select up to 5 predicted images
imgs = glob.glob(f"{pred_dir}/*.jpg")
assert imgs, f"No .jpg files found in: {pred_dir}"

k = min(5, len(imgs))
files = np.random.choice(imgs, size=k, replace=False)
print(files.shape)
print("Using:", pred_dir)

for image_path in files:
    display(Image(filename=image_path, height=600))
    print()

In [ ]:


# Wait for 30 seconds (e.g., to ensure all background tasks finish before disconnecting)
time.sleep(30)

# Gracefully disconnect the current Colab runtime session
runtime.unassign()


## 🏆 Congratulations

### Find more learning resources here

Roboflow has produced many resources that you may find interesting as you advance your knowledge of computer vision:

- [Roboflow Notebooks](https://github.com/roboflow/notebooks): A repository of over 20 notebooks that walk through how to train custom models with a range of model types, from YOLOv7 to SegFormer.
- [Roboflow YouTube](https://www.youtube.com/c/Roboflow): Our library of videos featuring deep dives into the latest in computer vision, detailed tutorials that accompany our notebooks, and more.
- [Roboflow Discuss](https://discuss.roboflow.com/): Have a question about how to do something on Roboflow? Ask your question on our discussion forum.
- [Roboflow Models](https://roboflow.com): Learn about state-of-the-art models and their performance. Find links and tutorials to guide your learning.

### Convert data formats

Roboflow provides free utilities to convert data between dozens of popular computer vision formats. Check out [Roboflow Formats](https://roboflow.com/formats) to find tutorials on how to convert data between formats in a few clicks.

### Connect computer vision to your project logic

[Roboflow Templates](https://roboflow.com/templates) is a public gallery of code snippets that you can use to connect computer vision to your project logic. Code snippets range from sending emails after inference to measuring object distance between detections.